# 🏛️ VietLens Vision Server (Qwen2.5-VL-3B + Vietnam Landmark LoRA)
Notebook này khởi chạy Server suy luận AI đa phương thức `Qwen/Qwen2.5-VL-3B-Instruct` kết hợp bộ trọng số tinh chỉnh LoRA mới cho 14 địa danh Đà Nẵng & Miền Trung trên GPU Google Colab (T4), tự động mở cổng kết nối Ngrok cho `VLensModel`.

### ⚡ Hướng dẫn nhanh:
1. Đảm bảo Runtime đã chọn **T4 GPU** (Runtime -> Change runtime type -> T4 GPU).
2. Nhấn **Run All** (hoặc Ctrl + F9). Notebook sẽ tự động kết nối Google Drive, nạp bộ trọng số LoRA mới và khởi động server qua Ngrok.
3. Copy đường link Ngrok được in ra ở Cell cuối cùng (ví dụ: `https://xxxx.ngrok-free.app`) và chạy trên máy tính:
   ```bat
   cd VLensModel
   set_vision_url.bat https://xxxx.ngrok-free.app
   run_model.bat
   ```


In [1]:
# 1. CÀI ĐẶT THƯ VIỆN CẦN THIẾT (TỐI ƯU HÓA SUY LUẬN TRÊN T4 GPU)
!pip install -q -U transformers accelerate peft bitsandbytes qwen-vl-utils
!pip install -q fastapi uvicorn pyngrok python-multipart nest_asyncio python-dotenv

# Tự động dọn dẹp các tiến trình chiếm dụng GPU chạy ngầm từ phiên trước
import os, signal, subprocess, gc, torch

def cleanup_gpu():
    try:
        out = subprocess.check_output('nvidia-smi --query-compute-apps=pid --format=csv,noheader', shell=True).decode()
        cur_pid = os.getpid()
        for p in out.strip().split('\n'):
            p = p.strip()
            if p and p.isdigit() and int(p) != cur_pid:
                try:
                    os.kill(int(p), signal.SIGKILL)
                except Exception:
                    pass
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

cleanup_gpu()

if torch.cuda.is_available():
    free_mem, total_mem = torch.cuda.mem_get_info()
    print(f'CUDA Available: True')
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')
    print(f'GPU VRAM: {free_mem / (1024**3):.2f} GB free / {total_mem / (1024**3):.2f} GB total')
else:
    print('⚠️ Chưa bật GPU! Hãy chọn Runtime -> Change runtime type -> T4 GPU.')


CUDA Available: True
GPU Device: Tesla T4
GPU VRAM: 14.46 GB free / 14.56 GB total


In [2]:
# 2. IMPORT CÁC THƯ VIỆN CẦN THIẾT
import io
import os
import re
import json
import zipfile
import threading
import asyncio
import math
from pathlib import Path
from PIL import Image
import torch
import torch.nn.functional as F
from fastapi import FastAPI, UploadFile, Form
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig
)
from qwen_vl_utils import process_vision_info
from peft import PeftModel

print('✅ Đã import toàn bộ thư viện thành công.')


✅ Đã import toàn bộ thư viện thành công.


## 🤖 3. Tự Động Nạp Bộ Trọng Số LoRA Mới & Khởi Tạo Model (4-bit)
Hệ thống sẽ tự động tìm kiếm file `qwen2_5_vl_danang_lora.zip` trong Google Drive của bạn, giải nén và nạp vào Base Model `Qwen2.5-VL-3B-Instruct` dạng 4-bit tiết kiệm VRAM.


In [3]:
LORA_DIR = Path('./qwen2_5_vl_danang_lora')
ZIP_NAME = 'qwen2_5_vl_danang_lora.zip'

# 1. Tự động kiểm tra và giải nén adapter LoRA mới
if not LORA_DIR.exists():
    # Kiểm tra file zip ở thư mục hiện tại của Colab
    if Path(ZIP_NAME).exists():
        print(f'📦 Đang giải nén {ZIP_NAME} từ thư mục hiện tại...')
        with zipfile.ZipFile(ZIP_NAME, 'r') as zf:
            zf.extractall(LORA_DIR)
    else:
        # Kiểm tra trong Google Drive
        try:
            from google.colab import drive
            if not Path('/content/drive/MyDrive').exists():
                print('⏳ Đang kết nối Google Drive...')
                drive.mount('/content/drive')
            
            drive_zip = Path(f'/content/drive/MyDrive/{ZIP_NAME}')
            if drive_zip.exists():
                print(f'📦 Tìm thấy adapter trên Google Drive: {drive_zip}')
                with zipfile.ZipFile(drive_zip, 'r') as zf:
                    zf.extractall(LORA_DIR)
            else:
                # Tìm kiếm quét trong toàn bộ thư mục con của MyDrive
                found = list(Path('/content/drive/MyDrive').glob(f'**/{ZIP_NAME}'))
                if found:
                    print(f'📦 Tìm thấy adapter tại: {found[0]}')
                    with zipfile.ZipFile(found[0], 'r') as zf:
                        zf.extractall(LORA_DIR)
        except Exception as e:
            print(f'ℹ️ Google Drive info: {e}')

# 2. Xác định đường dẫn LoRA hợp lệ
if LORA_DIR.exists():
    LORA_PATH = str(LORA_DIR)
    print(f'🎯 SỬ DỤNG LORA ADAPTER MỚI VỪA TRAIN: {LORA_PATH}')
else:
    LORA_PATH = 'imbee510/qwen2-5-vl-landmark-vietnam-lora'
    print(f'⚠️ Không tìm thấy file zip cục bộ, fallback về HuggingFace: {LORA_PATH}')

BASE_MODEL = 'Qwen/Qwen2.5-VL-3B-Instruct'

import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Cấu hình 4-bit NF4 giúp tiêu thụ chỉ ~2.5GB VRAM, tốc độ suy luận dưới 1 giây
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f'⏳ Đang tải Base Model: {BASE_MODEL} (4-bit)...')
# device_map={"": 0} ép toàn bộ mô hình nạp vào GPU 0, ngăn accelerate tự ý đẩy sang CPU
target_device = {'': 0} if torch.cuda.is_available() else 'auto'
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    quantization_config=bnb_config,
    device_map=target_device
)

print(f'⏳ Đang tải LoRA weights: {LORA_PATH}...')
model = PeftModel.from_pretrained(base_model, LORA_PATH)
model.eval()

# Khống chế kích thước ảnh để suy luận nhanh và mượt mà
min_pixels = 256 * 28 * 28
max_pixels = 512 * 28 * 28
processor = AutoProcessor.from_pretrained(
    BASE_MODEL,
    min_pixels=min_pixels,
    max_pixels=max_pixels
)

print('✅ Khởi tạo Vision Model và LoRA Adapter thành công!')


🎯 SỬ DỤNG LORA ADAPTER MỚI VỪA TRAIN: qwen2_5_vl_danang_lora
⏳ Đang tải Base Model: Qwen/Qwen2.5-VL-3B-Instruct (4-bit)...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

⏳ Đang tải LoRA weights: qwen2_5_vl_danang_lora...
✅ Khởi tạo Vision Model và LoRA Adapter thành công!


## 🧠 4. Tiền Xử Lý Ảnh & Parser Đầu Ra Chuẩn Của VietLens
Hàm phân giải kết quả bóc tách chuẩn xác tên địa danh, thành phố và điểm tin cậy để gửi về cho Backend.


In [4]:
TRAINING_SYSTEM_PROMPT = 'You are a highly capable AI assistant specialized in identifying global landmarks.'
TRAINING_USER_QUESTION = 'What is the name of the place in this photo?'

def pad_to_square(img):
    w, h = img.size
    size = max(w, h)
    new_img = Image.new('RGB', (size, size), (0, 0, 0))
    new_img.paste(img, ((size - w) // 2, (size - h) // 2))
    return new_img

def smart_resize_by_tokens(img, target_tokens=196, patch_size=28):
    w, h = img.size
    target_grid = int(math.sqrt(target_tokens))
    scale = min((target_grid * patch_size) / max(w, h), 1.0)
    new_w = max(patch_size, (int(w * scale) // patch_size) * patch_size)
    new_h = max(patch_size, (int(h * scale) // patch_size) * patch_size)
    return img.resize((new_w, new_h), Image.Resampling.LANCZOS)

def preprocess_for_lora(img):
    img = img.convert('RGB')
    img = pad_to_square(img)
    img = smart_resize_by_tokens(img, target_tokens=196, patch_size=28)
    return img

def parse_lora_output(text: str):
    text_clean = text.strip()
    if 'outside scope' in text_clean.lower() or 'out_of_domain' in text_clean.lower():
        return {
            'landmark_name': 'OUT_OF_DOMAIN',
            'city': '',
            'country': 'Vietnam',
            'vision_reasoning': text_clean,
            'alternative_candidates': []
        }
    # Parse dạng JSON nếu model trả về JSON
    if text_clean.startswith('{'):
        try:
            return json.loads(re.sub(r'```json|```', '', text_clean).strip())
        except Exception:
            pass
    # Parse chuẩn format LoRA: 'Based on the visual features, this is [Landmark], [City].'
    prefix = 'Based on the visual features, this is '
    body = text_clean
    if prefix.lower() in body.lower():
        idx = body.lower().find(prefix.lower()) + len(prefix)
        body = body[idx:].strip()
    body = body.rstrip('.')
    parts = [p.strip() for p in body.split(',') if p.strip()]
    landmark_name = parts[0] if parts else 'Unknown'
    city = parts[1] if len(parts) > 1 else 'Vietnam'
    return {
        'landmark_name': landmark_name,
        'city': city,
        'country': 'Vietnam',
        'vision_reasoning': text_clean,
        'alternative_candidates': []
    }

class VisionModel:
    def __init__(self):
        self.model = model
        self.processor = processor
        self.model.eval()

    def predict(self, prompt, image_path):
        processed_img = preprocess_for_lora(image_path)
        messages = [
            {'role': 'system', 'content': [{'type': 'text', 'text': TRAINING_SYSTEM_PROMPT}]},
            {
                'role': 'user',
                'content': [
                    {'type': 'image', 'image': processed_img},
                    {'type': 'text', 'text': TRAINING_USER_QUESTION}
                ]
            }
        ]

        text = self.processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = self.processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors='pt'
        ).to('cuda')

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                output_scores=True,
                return_dict_in_generate=True
            )

        generated_ids = outputs.sequences
        try:
            input_len = inputs.input_ids.shape[1]
        except AttributeError:
            input_len = inputs['input_ids'].shape[1]

        gen_logits = torch.stack(outputs.scores, dim=1)
        gen_probs = F.softmax(gen_logits, dim=-1)
        generated_tokens = generated_ids[:, input_len:]
        token_probs = torch.gather(gen_probs, 2, generated_tokens.unsqueeze(-1)).squeeze(-1)
        real_confidence = float(token_probs.mean().item()) if token_probs.numel() > 0 else 0.95

        generated_ids_trimmed = [
            out_ids[len(in_ids):]
            for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]

        output_text = self.processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )[0]

        return output_text, real_confidence

vision_model = VisionModel()


## 🌐 5. Khởi Tạo FastAPI Web Server (Cổng 8001)
Định nghĩa các endpoint `/predict_landmark` và `/health` phục vụ giao tiếp với Backend `VLensModel`.


In [5]:
app = FastAPI(title='VietLens Vision Server', version='2.0')

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

@app.get('/')
@app.get('/health')
def health_check():
    return {
        'status': 'healthy',
        'service': 'VietLens Vision Server',
        'base_model': BASE_MODEL,
        'lora_adapter': LORA_PATH,
        'device': 'cuda' if torch.cuda.is_available() else 'cpu'
    }

@app.post('/predict_landmark')
async def detect(file: UploadFile, prompt: str = Form('')):
    image_data = await file.read()
    image = Image.open(io.BytesIO(image_data))

    result_text, real_confidence = vision_model.predict(
        prompt=prompt,
        image_path=image
    )

    print('\n[Vision Server] Model Raw Output:\n', result_text)

    parsed = parse_lora_output(result_text)
    parsed['reasoning_confidence'] = real_confidence
    parsed['raw_output'] = result_text
    return parsed


## 🚀 6. Mở Kênh Ngrok & Bật Server Chạy Ngầm
Sử dụng `NGROK_TOKEN` từ file `.env` để tạo đường link Public URL an toàn truy cập từ máy tính của bạn.


In [6]:
import uvicorn
from dotenv import load_dotenv
load_dotenv()

# Đóng các kết nối Ngrok cũ trước đó nếu có
try:
    ngrok.kill()
except Exception:
    pass

# Lấy NGROK_TOKEN từ biến môi trường (.env) hoặc Colab Secrets
NGROK_TOKEN = os.getenv('NGROK_TOKEN', '')
if not NGROK_TOKEN:
    try:
        from google.colab import userdata
        NGROK_TOKEN = userdata.get('NGROK_TOKEN') or ''
    except Exception:
        NGROK_TOKEN = ''

if NGROK_TOKEN and NGROK_TOKEN.strip():
    ngrok.set_auth_token(NGROK_TOKEN.strip())
else:
    print('⚠️ Vui lòng cung cấp NGROK_TOKEN hợp lệ!')

public_tunnel = ngrok.connect(8001)
public_url = public_tunnel.public_url

print('=' * 65)
print(f'🚀 YOUR NGROK URL: {public_url}')
print('=' * 65)
print('👉 CÁC BƯỚC TIẾP THEO TRÊN MÁY TÍNH CỦA BẠN:')
print('   1. Mở Terminal trong thư mục VLensModel')
print(f'   2. Chạy: .\\set_vision_url.bat {public_url}')
print('   3. Chạy: .\\run_model.bat')
print('   4. Mở app VLensFE và quét ảnh thử nghiệm ngay!')
print('=' * 65)

def run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(app, host='0.0.0.0', port=8001, log_level='info')
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
print('✅ Vision Server is running in the background on port 8001!')


PyngrokNgrokError: An error occurred when saving the auth token: b'Authtoken already set in configuration file: /root/.config/ngrok/ngrok.yml\n'